# 📗 DeepEval: 지표와 테스트 케이스

## 오늘 배울 내용

교안 01에서는 판사 프롬프트를 직접 만들어 답변을 채점하고, 판사 판정을 사람 판정과 대조했습니다. 이번에는 같은 평가를 **DeepEval**로 옮겨, 평가 사례와 지표를 코드로 정리하고 여러 사례를 한 번에 실행합니다.

**DeepEval이란?** 답변 평가를 파이썬 코드로 반복 실행하고 결과를 모아 주는 평가 도구입니다. 평가 기준은 사람이 정하고, 판사 모델이 그 기준으로 답변을 판정합니다.

**먼저 알아둘 말**

| 용어 | 뜻 | DeepEval 코드 |
|---|---|---|
| 평가 사례 | 질문·평가할 답·기준 답변·근거를 한데 모은 한 건 | `LLMTestCase` |
| 평가 지표 | 답변을 어떤 기준으로 채점할지 정한 방법 | `GEval`, `HallucinationMetric` 등 |
| 실행 | 여러 사례에 지표를 적용하고 결과를 모음 | `evaluate()` |

**오늘의 순서**

| 단계 | 하는 일 | 절 |
|---|---|---|
| ① 평가 사례 만들기 | 질문·답변·기준 답변·근거를 `LLMTestCase` 한 건으로 묶기 | 3절 |
| ② 기준을 지표로 적기 | G-Eval로 판단 단계·rubric을 정하고 점수와 판정 이유 읽기 | 4절 |
| ③ 목적별 지표 쓰기 | 근거와의 모순, 무례한 표현, 집단 고정관념을 전용 지표로 평가 | 5~6절 |
| ④ 사람 판정과 대조하기 | 평가 축마다 알맞은 방법(G-Eval·코드·전용 지표)을 골라 판사 점검 | 7절 |
| ⑤ 묶어서 평가하기 | 새 답변을 만들고 `evaluate()`로 함께 채점한 뒤 실패 이유 읽기 | 7~8절 |

9절 **함께 따라하기**에서는 같은 방법으로 상담 챗봇 답변을 정책 기준으로 평가합니다.

**실무에서 쓰는 방법:** 사람이 검토한 대표 질문과 기준 답변을 모아 두고, 모델이나 프롬프트를 바꿀 때 같은 사례를 다시 평가합니다. 형식처럼 규칙이 분명한 항목은 코드로 확인하고, 말투나 정책 적용처럼 맥락이 필요한 항목은 판사 모델의 판정을 사람이 확인합니다. 여러 요청을 동시에 보내면 대기 시간을 줄일 수 있지만 요청 횟수와 비용은 줄지 않습니다.

**RAG와의 관계:** 검색 증강 생성(RAG) 답변에도 같은 평가 방법을 쓸 수 있습니다. 이 교안에서는 검색을 실행하지 않고 미리 준비한 근거로 답변을 평가합니다. RAGAS와 DeepEval 모두 RAG와 일반 응답 평가에 사용할 수 있습니다. [RAGAS 공식 지표 목록](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/) · [DeepEval 공식 지표 소개](https://deepeval.com/docs/metrics-introduction)

<img src="./images/00_lesson02_goals.png" width="720" alt="오늘의 학습 목표 네 가지: 평가 사례 만들기, 기준에 따라 채점하기, 지표를 나눠 보기, 여러 사례를 함께 평가하기">

**학습 목표**

1. 질문·평가할 답·기준 답변·근거를 `LLMTestCase` 한 건에 담습니다.
2. G-Eval 기준을 작성하고, 기준 답변(`expected_output`)을 넣었을 때와 뺐을 때의 판정을 비교합니다. 기준 답변을 줘도 오판은 남을 수 있습니다.
3. 제공 근거와의 모순, 무례한 표현, 집단 고정관념을 별도 지표로 평가합니다. DeepEval 4.1.1에서는 세 지표 모두 점수가 낮을수록 좋습니다.
4. 평가 축마다 알맞은 지표를 고르고, `evaluate()`로 여러 사례를 묶어 평가한 뒤 사람 판정과 대조합니다.

## 1. DeepEval로 반복 평가하기

교안 01에서는 LangChain 체인으로 판사 모델에게 기준을 주고 답을 채점했습니다. DeepEval은 평가를 파이썬 코드로 반복 실행하고 결과를 모아 보는 도구입니다. 평가 사례를 같은 형식으로 준비하고, 한 개 또는 여러 개의 지표를 적용할 수 있습니다.

<img src="./images/11_framework.png" width="760" alt="평가 사례에 질문·답·근거를 담고, 지표를 골라 evaluate로 실행한 뒤 케이스별 결과를 확인하는 흐름">

DeepEval로 답변을 평가하는 흐름은 다음과 같습니다.

1. 질문·평가할 답·필요한 근거를 평가 사례 한 건으로 묶습니다.
2. 판정 기준이나 목적에 맞는 지표를 고릅니다.
3. 여러 사례와 지표를 `evaluate()`로 함께 평가합니다.
4. 각 사례의 점수·통과 여부·판정 이유를 읽습니다.

## 2. 실습 준비

### 2-1. 실습 대상: 루나와 루나 굿즈 스토어

<table>
<tr>
<td width="250" valign="top" align="center">
<img src="./images/01_luna_character.png" width="230" alt="별 장식과 분홍 리본, 흰색·분홍색 무대 의상과 마이크를 든 루나">
<br><sub>수업용 가상 캐릭터 · 루나</sub>
</td>
<td valign="top">

**루나**는 팬과 대화하는 **수업용 버추얼 아이돌 챗봇**입니다. 교안 01과 같은 설정을 씁니다.

- 따뜻하고 경쾌한 말투로 팬을 응원합니다.
- 플레이리스트 질문과 가상 팬미팅 안내에 답합니다.
- 설정과 공지는 `data/luna_guide.json`의 L1~L8 문장에 있습니다.

**루나 굿즈 스토어 상담 챗봇**(9절)은 응원봉·포토카드 같은 가상 굿즈의 교환·반품 질문에 정책(P1~P5)대로 답합니다.

등장하는 인물·설정·일정·정책은 모두 실습용 가상 자료입니다.

</td>
</tr>
</table>

| 평가 축 | 확인할 점 | 이 교안의 평가 방법 |
|---|---|---|
| 페르소나 일관성 | 루나의 설정과 말투를 지키는가 | G-Eval(7-1) |
| 질문 충족도·근거성 | 질문에 답하고, 공지와 맞으며 없는 정보를 만들지 않는가 | G-Eval(4절, 7-3), Hallucination(5절) |
| 형식 준수 | 지정한 줄 수와 항목 순서를 지키는가 | 코드 검사(7-2) |
| 존중 표현·편향 | 팬을 모욕하지 않고, 성별로 취향을 단정하지 않는가 | Toxicity·Bias(6절) |
| 정책 준수 | 정책 조건을 빠짐없이 안내하고 없는 약속을 하지 않는가 | G-Eval(9절) |

### 2-2. 답변 모델과 판사 모델 설정

답변 모델은 챗봇 답을 만들고, 판사 모델은 그 답을 평가합니다. 역할을 나누면 자기 답을 더 좋아하는 편향의 위험을 낮출 수 있습니다. 설정 셀에서는 답변 모델을 `answer_llm`, 판사 모델을 `judge_llm`이라고 부릅니다.

DeepEval에 전달하는 판사 객체는 `GPTModel`입니다. 코드 이름은 교안 01과 같지만, 여기서는 DeepEval에서 사용하는 모델 객체를 담습니다.

DeepEval 4.1.1의 G-Eval은 모델이 다음 글자·단어 조각을 고를 때의 확률 정보(`logprobs`)를 사용합니다. 이 교안은 공식 지원 예시인 `gpt-4.1`을 기본 판사 모델로 둡니다. 다른 모델로 바꾸면 `logprobs` 지원 여부를 확인합니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
import json
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import pandas as pd

# 표에서 이유 같은 긴 문장이 잘리지 않게 합니다.
pd.set_option("display.max_colwidth", None)

# DeepEval이 사용 기록을 외부로 보내지 않게 합니다. deepeval을 불러오기 전에 정해야 합니다.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"
import deepeval
from deepeval.models import GPTModel
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig

# material_dir는 data가 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"

# 단원 폴더에서 가까운 .env를 찾아 API 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 상위 실습자료 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다.
answer_model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
answer_llm = ChatOpenAI(model=answer_model_name, use_responses_api=True)
print("답변 모델:", answer_model_name)

# 판사 모델: 채점은 답변을 만든 모델과 다른 모델에게 맡깁니다. OPENAI_EVAL_MODEL 환경 변수로 선택합니다.
judge_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-4.1")
# temperature=0: 출력 변동을 낮추려는 설정이며, 같은 판정을 보장하지는 않습니다.
judge_llm = GPTModel(model=judge_model_name, temperature=0)
print("판사 모델:", judge_model_name)
print("DeepEval 버전:", deepeval.__version__)


# [보조 함수] 자료 읽기: data 폴더의 JSON 파일을 읽어 파이썬 객체로 반환합니다.
def read_json(name):
    """data 폴더의 JSON 파일을 읽어 파이썬 객체로 반환합니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))

### 2-3. 실습 자료 살펴보기

| 파일 | 들어 있는 내용 | 쓰는 곳 |
|---|---|---|
| `luna_guide.json` | 루나 설정과 팬미팅 공지 문장(L1~L8), 평가에 쓸 정보의 범위 | 전체 |
| `luna_metric_cases.json` | 지표를 연습할 루나 답변 8건. 각 사례에 주된 결함 한 가지(`defect`) | 3~5절 |
| `luna_pairwise_cases.json` | 두 답을 비교할 루나 사례 6건(이 교안은 LP04만 사용) | 4-4 |
| `luna_eval_cases.json` | 페르소나·질문 충족·형식·근거성·존중 표현·편향 사례 15건과 사람 판정 | 6~7절 |
| `luna_coverage_cases.json` | 평가 범위표에서 고른 새 질문 6건 | 7-5~8절 |
| `store_policies.json` 등 `store_*.json` | 굿즈 스토어의 가상 정책 5개, 사람 판정이 있는 상담 기록 8건(답 16개), 새 질문 6건 | 9절 |

DeepEval의 `LLMTestCase`는 정해진 필드 이름을 씁니다. 평가할 답변은 `actual_output`, 사람이 확인한 기준 답변은 `expected_output`, 답변에 필요한 근거 문장은 `context`에 넣습니다.

> **`expected_output`에 들어가는 두 종류의 기준:** 이 자료에는 그대로 비교할 모범 답 문장(예: `event_ok`의 "2026년 11월 8일 오후 7시에 루미너스홀에서 시작한다.")도 있고, 기대하는 행동을 설명한 문장(예: `ok`의 "따뜻하고 경쾌한 루나의 말투로 응원한다.")도 있습니다. 정답이 하나로 정해지지 않는 응원·페르소나 질문에는 행동 설명이 더 알맞습니다. 이 교안에서는 둘을 함께 **기준 답변**이라고 부르며, 어느 쪽이든 판사가 기대치를 알 수 있게 해 줍니다.

In [ ]:
# 루나의 설정과 가상 공지 문장을 L1~L8 ID로 보관합니다.
guide = read_json("luna_guide.json")
sentence_by_id = {}
for sentence in guide["sentences"]:
    sentence_by_id[sentence["id"]] = sentence["text"]
    print(sentence["id"] + ":", sentence["text"])
print("\n자료 범위:", guide["evidence_scope"])

In [ ]:
# 한 답변에 한 주된 결함을 둔 루나 예시를 읽습니다.
luna_metric_rows = read_json("luna_metric_cases.json")

# 케이스 이름으로 결함을 찾으면 지표별 결과를 사례와 연결할 수 있습니다.
defect_by_id = {}
case_rows = []
for case in luna_metric_rows:
    defect_by_id[case["id"]] = case["defect"]
    case_rows.append({"케이스 ID": case["id"], "주 결함": case["defect"],
                      "질문": case["input"], "답변": case["actual_output"],
                      "기준 답변": case["expected_output"], "근거 ID": ", ".join(case["context_ids"])})
display(pd.DataFrame(case_rows))

## 3. 평가 사례 한 건을 담는 `LLMTestCase`

**평가 사례**는 질문·평가할 답변·필요한 기준 답변·근거를 한데 모은 자료입니다. DeepEval에서는 사례 한 건을 `LLMTestCase`로 표현합니다.

<img src="./images/12_testcase_fields.png" width="760" alt="LLMTestCase: input은 질문, actual_output은 평가할 답변, expected_output은 사람이 확인한 기준 답변, context는 미리 준비한 근거 문장, retrieval_context는 검색기가 실제 찾아온 문서입니다. 이번 실습에서는 검색기를 실행하지 않습니다.">

루나가 팬미팅 시작 시각을 잘못 안내한 사례를 보면:

1. `input`에는 팬이 한 질문을 넣습니다.
2. `actual_output`에는 평가할 루나의 답변을 넣습니다.
3. `expected_output`에는 사람이 확인한 기준 답변을 넣습니다.
4. `context`에는 공지에서 고른 근거 문장을 넣습니다.

| 코드 필드 | 담는 내용 |
|---|---|
| `input` | 루나에게 한 질문 |
| `actual_output` | 평가할 루나의 답변 |
| `expected_output` | 사람이 확인한 기준 답변. 지표에 따라 생략 가능 |
| `context` | 평가 전에 준비한 근거 문장 목록 |
| `retrieval_context` | RAG 검색기가 실제로 찾아온 문서 목록. 이 실습에서는 사용하지 않음 |

평가 지표마다 필요한 필드가 다릅니다. `SingleTurnParams`는 G-Eval에게 어떤 필드를 보여 줄지 지정하는 코드 값입니다.

In [ ]:
from deepeval.test_case import LLMTestCase, SingleTurnParams

# 근거 ID를 실제 문장으로 바꿔 모든 예시를 LLMTestCase로 만듭니다.
luna_test_cases = []
test_case_by_id = {}
for case in luna_metric_rows:
    context_sentences = []
    for sentence_id in case["context_ids"]:
        context_sentences.append(sentence_by_id[sentence_id])

    test_case = LLMTestCase(
        input=case["input"],
        actual_output=case["actual_output"],
        expected_output=case["expected_output"],
        context=context_sentences,
        name=case["id"],
    )
    luna_test_cases.append(test_case)
    test_case_by_id[case["id"]] = test_case

# 일정 오류 사례에서 평가할 답과 사람이 확인한 답을 나란히 봅니다.
print("actual_output:", test_case_by_id["event_wrong"].actual_output)
print("expected_output:", test_case_by_id["event_wrong"].expected_output)
print("context:", test_case_by_id["event_wrong"].context)

**확인할 점:** `actual_output`의 오후 8시와 `expected_output`의 오후 7시가 다르고, `context`에는 공지 문장이 목록으로 들어 있는지 확인합니다. 사람의 기준 답변과 근거는 다른 역할을 합니다.

## 4. G-Eval로 평가 기준을 구체화하기

### 4-1. 판사 모델이 따를 단계를 정하기

**G-Eval**은 사람이 정한 기준을 판단 순서로 나누고, 판사 모델이 단계에 따라 점수와 판정 이유를 내게 하는 평가 지표입니다. 기준 문장만 주어도 단계가 만들어질 수 있고, 직접 `evaluation_steps`를 작성하면 판단할 순서를 구체적으로 정할 수 있습니다. 점수와 판정 이유는 모델을 실행할 때마다 달라질 수 있습니다. [DeepEval G-Eval 공식 문서](https://deepeval.com/docs/metrics-llm-evals)

<img src="./images/13_geval_flow.png" width="760" alt="루나 팬미팅 답변을 평가 기준과 판단 순서로 나누고, 판사 모델이 점수와 판정 이유를 내는 흐름">

**서비스 평가 기준**은 서비스에서 어떤 답변을 통과시키고 실패로 볼지 정한 규칙입니다. 루나의 팬미팅 안내라면 질문에 답했는지, 날짜·시간이 공지와 맞는지, 공지에 없는 정보를 만들어 내지 않았는지 살핍니다.

| 설정 | 쉬운 뜻 | 이 예에서 하는 일 |
|---|---|---|
| `name` | 지표 이름 | 어떤 기준의 결과인지 표시 |
| `evaluation_steps` | 판사가 따를 판단 순서 | 질문·근거·추가 정보를 차례로 확인 |
| `evaluation_params` | 판사에게 보여 줄 평가 사례의 칸 | 질문·답변·근거를 선택 |
| `model` | 평가에 쓸 판사 모델 | `judge_llm` |
| `threshold` | 통과 기준값 | 0.7은 실습용 출발값이며 사람 판정과 비교해 조정 |

이 G-Eval 점수는 0~1 사이입니다. `threshold`는 결과에 통과 표시를 할 경계값입니다. 높은 값이 곧 서비스에 적합하다는 보증은 아닙니다.

In [ ]:
from deepeval.metrics import GEval

# 루나 답변이 질문에 답하고 준비된 근거와 맞는지 평가합니다.
accuracy_steps = [
    "input에서 사용자가 묻는 핵심을 확인한다.",
    "actual_output이 input의 질문에 직접 답했는지 확인한다.",
    "actual_output의 일정·사실이 context와 일치하는지 확인한다.",
    "context가 뒷받침하지 않는 구체 사실을 단정하면 감점한다.",
    "말투와 길이는 이 지표의 점수에 반영하지 않는다.",
]

# expected_output 없이 질문·답·근거만으로 평가합니다.
accuracy_metric = GEval(
    name="질문 충족과 근거성",
    evaluation_steps=accuracy_steps,
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    model=judge_llm,
    threshold=0.7,
)

In [ ]:
# 미리 준비한 루나 응답을 하나씩 G-Eval로 채점합니다.
# measure()는 사례 한 건을 바로 채점합니다. 여러 사례를 동시에 실행하는 evaluate()는 6절부터 씁니다.
accuracy_rows = []
accuracy_score_by_id = {}
for case in luna_test_cases:
    accuracy_metric.measure(case)
    score = round(accuracy_metric.score, 2)
    accuracy_score_by_id[case.name] = score
    accuracy_rows.append({
        "케이스 ID": case.name,
        "주 결함": defect_by_id[case.name],
        "점수": score,
        "통과": accuracy_metric.success,
        "이유": accuracy_metric.reason,
    })
display(pd.DataFrame(accuracy_rows))

**확인할 점:** `under_answer`는 질문에 답하지 않고 응원만 하고, `event_wrong`은 공지와 다른 시간을 말하며, `unknown_detail`은 공지에 없는 VIP 입장 시간을 단정합니다. 세 사례는 이 지표가 겨냥한 결함(질문 충족·근거)이므로 낮은 점수가 나와야 합니다. 판사가 놓칠 수도 있으므로 점수와 `이유`를 함께 읽습니다.

`rude`·`stereotype`의 주 결함(모욕·고정관념)은 이 지표의 평가 단계에 들어 있지 않습니다. `persona_drift`는 근거 문장 L1과 모순되므로 근거 확인 단계에서 잡힐 수 있지만, 말투 같은 페르소나 결함은 이 지표로 보지 않습니다. 이 사례들의 점수를 보면 **지표가 겨냥하지 않은 결함은 놓치거나 우연히만 잡힐 수 있다**는 점을 확인할 수 있습니다. 그래서 6~7절에서 목적에 맞는 지표를 따로 씁니다. 점수와 이유는 실행마다 달라질 수 있습니다.

### 4-2. 사람이 확인한 기준 답변을 함께 보기

`expected_output`은 사람이 미리 확인한 기준 답변을 담는 칸입니다. 기준 답변을 넣으면 판사가 답변의 핵심 내용을 비교할 예시를 갖게 됩니다. 이번에는 같은 답을 기준 답변 없이 평가한 결과와, 기준 답변을 함께 줬을 때의 결과를 나란히 봅니다.

기준 답변과 표현이 다르더라도 같은 정보를 전달하고 평가 기준을 지키면 통과할 수 있어야 합니다. 기준 답변 자체가 잘못됐을 수도 있으므로 결과만 믿지 말고 사람이 확인합니다.

In [ ]:
# 사람이 확인한 기준 답변과 context를 함께 읽는 G-Eval입니다.
accuracy_ref_metric = GEval(
    name="질문 충족과 근거성(기준 답변 포함)",
    evaluation_steps=[
        "input에서 사용자가 묻는 핵심을 확인한다.",
        "actual_output이 질문에 답하고 필요한 정보를 포함하는지 살핀다.",
        "actual_output의 핵심 내용이 expected_output 및 context와 맞는지 확인한다.",
        "context가 뒷받침하지 않는 구체 사실을 단정하면 감점한다.",
        "문장을 그대로 복사하지 않아도 같은 기준을 충족하면 통과로 본다.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                       SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    model=judge_llm,
    threshold=0.7,
)

# 같은 케이스를 기준 답변 없이, 있을 때로 나눠 비교합니다.
compare_rows = []
for case in luna_test_cases:
    accuracy_ref_metric.measure(case)
    compare_rows.append({
        "케이스 ID": case.name,
        "주 결함": defect_by_id[case.name],
        "기준 답변 없이": accuracy_score_by_id[case.name],
        "기준 답변 포함": round(accuracy_ref_metric.score, 2),
        "기준 답변 포함 이유": accuracy_ref_metric.reason,
    })
display(pd.DataFrame(compare_rows))

**확인할 점:** 각 케이스의 두 점수와 `기준 답변 포함 이유`를 읽고, 기준 답변이 판정에 어떤 정보를 보탰는지 확인합니다. `event_wrong`처럼 기준 답변이 모범 답 문장인 사례와, `ok`처럼 기대 행동 설명인 사례에서 기준 답변이 하는 역할을 비교해 봅니다. 기준 답변을 넣어도 판사의 오판 가능성은 남으며, 주관적인 기준은 사람 판정과 비교해야 합니다.

### 4-3. 점수 기준표로 점수 구간 설명하기

**점수 기준표(rubric)**는 점수마다 어떤 답변인지 미리 설명한 표입니다. 이 예에서는 팬미팅 안내가 공지와 얼마나 맞는지 점수 구간으로 나눕니다. 구간이 서로 겹치지 않도록 정합니다.

| 점수 구간 | 기대 결과 |
|---|---|
| 0~3 | 핵심 안내가 공지와 모순됨 |
| 4~7 | 핵심 일정은 맞지만 확인되지 않은 구체 정보가 있음 |
| 8~10 | 질문에 답하고 제공된 공지 범위 안에서 안내함 |

rubric 점수는 0~10 구간으로 적지만, 결과의 `score`는 다른 G-Eval처럼 0~1로 바뀌어 표시됩니다(예: 7점 → 0.7). 이 예는 8~10 구간만 통과로 보도록 `threshold`를 0.8로 둡니다.

`unknown_detail` 사례를 평가해 점수와 판정 이유가 표에 적은 기준에 맞는지 확인합니다.

In [ ]:
from deepeval.metrics.g_eval import Rubric

# rubric이 구간의 뜻을 보태는 효과를 한 사례로 살펴봅니다.
accuracy_rubric_metric = GEval(
    name="루나 안내 평가(rubric)",
    evaluation_steps=accuracy_steps,
    rubric=[
        Rubric(score_range=(0, 3), expected_outcome="핵심 안내가 공지와 모순된다."),
        Rubric(score_range=(4, 7), expected_outcome="핵심 안내는 맞지만 확인되지 않은 구체 사실이 있다."),
        Rubric(score_range=(8, 10), expected_outcome="질문에 답하고 공지 범위 안에서 안내한다."),
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    model=judge_llm,
    threshold=0.8,
)

accuracy_rubric_metric.measure(test_case_by_id["unknown_detail"])
print("점수:", round(accuracy_rubric_metric.score, 2))
print("통과:", accuracy_rubric_metric.success)
print("이유:", accuracy_rubric_metric.reason)

**확인할 점:** 점수 구간과 `이유`가 `unknown_detail`의 결함을 제대로 설명하는지 봅니다. `threshold`는 통과 여부만 바꾸며 판사의 판단 정확도를 자동으로 높이지 않습니다. 점수와 `threshold`가 가까우면 실행에 따라 통과 표시가 바뀔 수 있습니다.

### 4-4. Pairwise 비교를 DeepEval로

`ArenaGEval`은 같은 질문에 대한 후보 응답을 평가 단계에 따라 비교하는 지표입니다. 이번 예시는 루나 팬미팅 시작 시각을 잘못 안내한 답과 공지에 맞는 답(LP04)을 비교합니다. 두 후보는 같은 질문, 기준 답변, 근거를 사용합니다.

| 비교 방식 | 결과와 주의점 |
|---|---|
| 직접 만든 Pairwise(교안 01) | 승자·패자·무승부를 기준에 맞춰 정의하고, 순서를 바꿔 두 번 물어 위치 효과를 직접 확인 |
| `ArenaGEval` | 후보 중 승자 이름을 돌려주며 무승부는 반환하지 않음. DeepEval 4.1.1은 후보 이름을 가짜 이름으로 가리고 제시 순서를 무작위로 섞어 판사에게 보여 줌 |

이름 가리기와 순서 섞기는 이름·위치 효과를 줄이는 장치이지만, 한 번의 결과가 확정 판단이 되지는 않습니다. 아래에서는 같은 비교를 두 번 실행해 승자가 유지되는지 확인합니다. [DeepEval 4.1.1 ArenaGEval 구현](https://github.com/confident-ai/deepeval/blob/v4.1.1/deepeval/metrics/arena_g_eval/utils.py)

In [ ]:
from deepeval.metrics import ArenaGEval
from deepeval.test_case import ArenaTestCase, Contestant

# 같은 질문의 사전 작성 후보 두 개를 가져옵니다.
arena_source = next(row for row in read_json("luna_pairwise_cases.json") if row["pair_id"] == "LP04")
arena_context = [sentence_by_id[sentence_id] for sentence_id in arena_source["context_ids"]]
# 기준 답변은 3절 event_ok 사례와 같은 문장을 씁니다.
arena_reference = test_case_by_id["event_ok"].expected_output

# 후보마다 질문·기준 답변·근거를 동일하게 둡니다.
arena_case = ArenaTestCase(contestants=[
    Contestant(name="시간 오류 답", test_case=LLMTestCase(
        input=arena_source["question"], actual_output=arena_source["answer_a"],
        expected_output=arena_reference, context=arena_context, name="후보 A")),
    Contestant(name="공지와 맞는 답", test_case=LLMTestCase(
        input=arena_source["question"], actual_output=arena_source["answer_b"],
        expected_output=arena_reference, context=arena_context, name="후보 B")),
])

# 기준 답변·공지 근거와의 일치로 후보를 비교합니다.
arena_metric = ArenaGEval(
    name="팬미팅 안내 비교",
    evaluation_steps=[
        "각 후보가 같은 질문에 답했는지 확인한다.",
        "답변의 날짜와 시간이 expected_output 및 context와 일치하는지 비교한다.",
        "더 정확하고 근거에 맞는 후보를 고른다.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                       SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    model=judge_llm,
)

# 같은 비교를 두 번 실행합니다. 실행마다 후보 제시 순서가 무작위로 섞입니다.
for run in [1, 2]:
    arena_metric.measure(arena_case)
    print(f"{run}회차 승자:", arena_metric.winner)
    print("  이유:", arena_metric.reason)

**확인할 점:** 두 번 모두 `공지와 맞는 답`이 승자인지, `이유`가 공지의 시작 시각(오후 7시)을 짚는지 봅니다. 두 실행의 승자가 다르면 판정이 제시 순서나 호출 변동에 흔들린 것이므로 확정 판단으로 쓰지 않습니다. 사람 선호(`human_winner`: B)와 비교하고, 중요한 비교는 여러 사례로 확인합니다.

## 5. 답변과 준비한 근거가 모순되는지 확인하기

DeepEval 4.1.1의 `HallucinationMetric`은 답변 내용과 미리 준비한 근거 문장(`context`)이 서로 모순되는지 평가합니다. 이 버전은 모순으로 본 비율을 점수로 냅니다. 점수가 낮을수록 좋고, 통과 기준값 이하이면 통과입니다. 이 점수는 답변의 모든 사실이 참인지 보증하지 않습니다. [DeepEval 4.1.1 구현](https://github.com/confident-ai/deepeval/blob/v4.1.1/deepeval/metrics/hallucination/hallucination.py)

<img src="./images/14_hallucination_check.png" width="760" alt="루나의 팬미팅 일정 답변을 미리 준비한 공지 문장과 비교해 모순을 살핀다. 검색기는 실행하지 않는다.">

1. 질문과 직접 관련된 공지 문장 목록을 준비합니다.
2. 판사 모델이 답변과 각 문장의 모순 여부를 살핍니다.
3. 정상 안내, 틀린 시간, 공지에 없는 VIP 입장 시간을 비교합니다.
4. 점수와 판정 이유를 읽고, 근거가 답변을 판단하기에 충분했는지 확인합니다.

공지에 없는 내용을 새로 약속하는 것은 근거와 직접 모순되지 않을 수도 있습니다. 이런 사례는 기준 답변이나 서비스에서 중요한 규칙을 함께 써서 평가합니다.

In [ ]:
from deepeval.metrics import HallucinationMetric

hallucination_metric = HallucinationMetric(model=judge_llm, threshold=0.5)
hallucination_rows = []
hallucination_verdicts = {}

# 세 루나 사례의 판정 이유를 케이스별로 따로 보관합니다.
for case_id in ["event_ok", "event_wrong", "unknown_detail"]:
    case = test_case_by_id[case_id]
    hallucination_metric.measure(case)
    hallucination_verdicts[case_id] = hallucination_metric.verdicts
    hallucination_rows.append({
        "케이스 ID": case.name,
        "주 결함": defect_by_id[case.name],
        "점수": round(hallucination_metric.score, 2),
        "통과": hallucination_metric.success,
    })
display(pd.DataFrame(hallucination_rows))

정상 일정(`event_ok`)·잘못된 시각(`event_wrong`)·확인되지 않은 VIP 입장 시간(`unknown_detail`)에서 공지 문장별 판정을 확인합니다.

In [ ]:
# 근거 문장과 판사 판정을 같은 순서로 펼쳐 이유를 확인합니다.
verdict_rows = []
for case_id in ["event_ok", "event_wrong", "unknown_detail"]:
    context_sentences = test_case_by_id[case_id].context
    verdicts = hallucination_verdicts[case_id]
    for sentence, verdict in zip(context_sentences, verdicts):
        verdict_rows.append({"케이스 ID": case_id, "공지 문장": sentence,
                             "판정": verdict.verdict, "이유": verdict.reason})
display(pd.DataFrame(verdict_rows))

**확인할 점:** `event_wrong`에서 공지의 시작 시각과 모순된다는 판정이 있는지 확인합니다. `unknown_detail`에는 "VIP 입장 시간은 공지에 없다"는 문장(L5)을 함께 넣었기 때문에 판사가 없는 사실을 판단할 단서를 얻습니다. 이런 문장이 없으면 공지에 없는 약속은 어떤 문장과도 모순되지 않아 낮은(좋은) 점수가 나올 수 있습니다. 반대로 `context`가 질문과 관련 없는 문장뿐이거나 비어 있을 때 나온 낮은 점수도 답변이 옳다는 뜻은 아닙니다.

In [ ]:
# 질문과 무관한 문장을 더 넣었을 때 판정이 달라지는지 비교합니다.
all_sentences = [sentence["text"] for sentence in guide["sentences"]]
event_case = test_case_by_id["event_ok"]
event_all_case = LLMTestCase(
    input=event_case.input,
    actual_output=event_case.actual_output,
    context=all_sentences,
)

hallucination_metric.measure(event_case)
print("관련 공지:", round(hallucination_metric.score, 2))
hallucination_metric.measure(event_all_case)
print("루나 안내 전체:", round(hallucination_metric.score, 2))

# 각 문장 판정에서 질문과 무관한 정보가 오해를 낳는지 확인합니다.
all_verdict_rows = []
for sentence, verdict in zip(all_sentences, hallucination_metric.verdicts):
    all_verdict_rows.append({"공지 문장": sentence, "판정": verdict.verdict, "이유": verdict.reason})
display(pd.DataFrame(all_verdict_rows))

**확인할 점:** 관련 공지만 넣었을 때와 안내 전체를 넣었을 때의 점수·판정 이유를 비교합니다. 답변이 말하지 않은 주변 정보만으로 모순 판정을 받았다면 context 구성이 부적절했을 수 있습니다. 근거는 많을수록 좋은 것이 아니라 질문과 관련되어야 합니다.

## 6. 존중 표현과 편향을 따로 평가하기

두 항목은 서로 다른 문제를 봅니다. **존중 표현**은 팬을 모욕하거나 위협하는지 살핍니다. **편향**은 성별 같은 집단 정보만으로 취향을 미리 단정하는지 봅니다. 친절하게 말하면서도 편향된 내용을 담을 수 있고, 편향 없이 말하면서도 무례할 수 있으므로 평가 기준을 나눕니다.

`ToxicityMetric`은 모욕·위협 같은 유해 표현을, `BiasMetric`은 편향 표현을 평가합니다. 두 지표 모두 점수가 낮을수록 좋습니다. DeepEval 4.1.1에서는 점수가 통과 기준값 0.5 이하일 때 통과로 표시합니다. 0.5는 실습용 출발값이며 서비스 품질 보증 기준은 아닙니다. [ToxicityMetric 4.1.1](https://github.com/confident-ai/deepeval/blob/v4.1.1/deepeval/metrics/toxicity/toxicity.py) · [BiasMetric 4.1.1](https://github.com/confident-ai/deepeval/blob/v4.1.1/deepeval/metrics/bias/bias.py)

In [ ]:
# 사람 판정이 있는 존중 표현·편향 사례를 읽습니다.
luna_eval_rows = read_json("luna_eval_cases.json")
social_eval_rows = [row for row in luna_eval_rows if row["dimension"] in ("toxicity", "bias")]
social_case_by_id = {row["case_id"]: row for row in social_eval_rows}

social_test_cases_by_dimension = {"toxicity": [], "bias": []}
for row in social_eval_rows:
    social_test_cases_by_dimension[row["dimension"]].append(LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"],
        expected_output=row["expected_output"],
        context=row["context"],
        name=row["case_id"],
    ))

display(pd.DataFrame([
    {"케이스 ID": row["case_id"], "평가 축": row["dimension"],
     "사람 판정": row["human_label"], "답변": row["actual_output"]}
    for row in social_eval_rows
]))

In [ ]:
from deepeval.metrics import ToxicityMetric, BiasMetric

# 각 평가 축에 맞는 지표만 같은 축의 답변에 적용합니다.
toxicity_metric = ToxicityMetric(model=judge_llm, threshold=0.5)
bias_metric = BiasMetric(model=judge_llm, threshold=0.5)

social_eval_rows_by_metric = []
for dimension, metric in [("toxicity", toxicity_metric), ("bias", bias_metric)]:
    result = evaluate(
        test_cases=social_test_cases_by_dimension[dimension],
        metrics=[metric],
        async_config=AsyncConfig(max_concurrent=2, throttle_value=0),
        display_config=DisplayConfig(print_results=False, show_indicator=False),
    )
    # evaluate()의 결과 순서에 기대지 않고 케이스 이름으로 사람 판정을 찾습니다.
    for test_result in result.test_results:
        metric_result = test_result.metrics_data[0]
        source_row = social_case_by_id[test_result.name]
        judge_label = "pass" if metric_result.success else "fail"
        social_eval_rows_by_metric.append({
            "케이스 ID": test_result.name,
            "평가 축": "존중 표현" if dimension == "toxicity" else "편향",
            "사람 판정": source_row["human_label"],
            "판사 판정": judge_label,
            "일치": source_row["human_label"] == judge_label,
            "점수": round(metric_result.score, 2),
            "판정 이유": metric_result.reason,
        })

display(pd.DataFrame(social_eval_rows_by_metric))

**확인할 점:** 두 지표 모두 점수가 **낮을수록 좋고**, 0.5 이하이면 통과로 표시됩니다. L13·L15처럼 사람 판정이 fail인 답이 높은 점수로 실패 처리됐는지, L12·L14처럼 pass인 답이 낮은 점수로 통과했는지 봅니다. 이 결과는 7-4에서 루나 15건 전체의 판사 점검표에 함께 모읍니다.

## 7. 루나 답변을 여러 기준으로 평가하기

검색 결과의 품질 외에도 챗봇 답변을 여러 기준으로 평가할 수 있습니다. 이번 사례 모음에는 페르소나 일관성, 질문 충족도, 형식 준수, 공지 근거성, 존중 표현, 편향 사례가 있습니다. 모두 사람이 미리 검토한 수업용 예시이며 실제 서비스 기록이 아닙니다.

| 평가 축 | 사례 수 | 확인할 점 |
|---|---:|---|
| 페르소나 일관성 | 3 | 루나의 말투와 버추얼 아이돌 설정을 지키는지 |
| 질문 충족도 | 3 | 질문에 직접 답하고 시작할 정보를 안내하는지 |
| 형식 준수 | 2 | 일정을 지정한 두 줄로 정리하는지 |
| 근거성 | 3 | 공지와 일치하고 없는 정보를 사실처럼 만들지 않는지 |
| 존중 표현 | 2 | 팬을 모욕하거나 깎아내리지 않는지 |
| 편향 | 2 | 성별만으로 취향을 단정하지 않는지 |

존중 표현·편향 4건은 6절에서 평가했습니다. 이어서 7-1 페르소나(G-Eval), 7-2 형식(코드 검사), 7-3 질문 충족·근거성(G-Eval)을 평가하고, 7-4에서 15건 전체를 사람 판정과 대조합니다. 질문 하나에 대한 답만 보므로 여러 번 대화해도 캐릭터가 유지되는지는 평가하지 않습니다.

In [ ]:
# 루나 평가 사례 15건을 사람 판정과 함께 읽습니다.
display(pd.DataFrame([
    {"케이스 ID": row["case_id"], "평가 축": row["dimension"],
     "사람 판정": row["human_label"], "질문": row["input"],
     "답변": row["actual_output"], "사람이 붙인 이유": row["reason"]}
    for row in luna_eval_rows
]))

### 7-1. 루나다운 말투와 설정을 확인하기

세 사례에서 루나다운 따뜻한 말투, 딱딱한 말투, 버추얼 아이돌 설정 위반을 골라 사람 판정과 나란히 봅니다. 대화 이력을 넣지 않았으므로, 여러 차례 대화에서 말투가 일관적인지는 다루지 않습니다.

In [ ]:
# 페르소나 일관성에서 확인할 말투와 캐릭터 설정을 평가 단계로 적습니다.
luna_persona_metric = GEval(
    name="루나 페르소나 일관성",
    evaluation_steps=[
        "input의 요청에 직접 응답했는지 확인한다.",
        "actual_output이 따뜻하고 경쾌하며 팬을 존중하는지 확인한다.",
        "격식체나 공문처럼 건조한 말투는 페르소나 위반으로 본다.",
        "루나가 버추얼 아이돌이라는 설정을 지키는지 확인한다.",
        "expected_output은 기준 예시로 참고하되 문장을 그대로 복사하지 않아도 된다.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                       SingleTurnParams.EXPECTED_OUTPUT],
    model=judge_llm,
    threshold=0.7,
)

# 사람 판정이 있는 페르소나 사례만 이 지표에 연결합니다.
luna_persona_by_id = {}
luna_persona_cases = []
for row in luna_eval_rows:
    if row["dimension"] == "persona":
        luna_persona_by_id[row["case_id"]] = row
        luna_persona_cases.append(LLMTestCase(
            input=row["input"], actual_output=row["actual_output"],
            expected_output=row["expected_output"], name=row["case_id"],
        ))

In [ ]:
# 세 케이스를 평가하고 이름으로 사람 판정과 연결합니다.
luna_persona_result = evaluate(
    test_cases=luna_persona_cases,
    metrics=[luna_persona_metric],
    async_config=AsyncConfig(max_concurrent=3, throttle_value=0),
    display_config=DisplayConfig(print_results=False, show_indicator=False),
)
luna_persona_rows = []
for test_result in luna_persona_result.test_results:
    metric_data = test_result.metrics_data[0]
    source_row = luna_persona_by_id[test_result.name]
    judge_label = "pass" if metric_data.success else "fail"
    luna_persona_rows.append({
        "케이스 ID": test_result.name,
        "사람 판정": source_row["human_label"],
        "판사 판정": judge_label,
        "일치": source_row["human_label"] == judge_label,
        "점수": round(metric_data.score, 2),
        "판정 이유": metric_data.reason,
    })
display(pd.DataFrame(luna_persona_rows))

**확인할 점:** 사람 판정과 다른 케이스에서 판정 이유를 읽고 기준 문장과 사례를 어떻게 보완할지 생각합니다. 세 사례만으로 판사의 성능을 확정하지 않습니다.

### 7-2. 분명한 형식 규칙은 코드로 검사하기

줄 수·항목 이름·순서처럼 규칙이 분명한 형식은 코드로 검사합니다. 이 코드는 형식만 확인하고 날짜·시간의 사실 여부는 판단하지 않습니다. 사실 값은 공지와 대조하는 별도 기준으로 평가합니다.

In [ ]:
# [보조 함수] 형식 검사: 정확히 두 줄이고 항목 이름이 정한 순서인지 확인합니다.
REQUIRED_LABELS = ["날짜", "시간·장소"]

def check_two_line_format(text):
    """답변이 두 줄이고 '날짜', '시간·장소' 항목이 이 순서로 있는지 확인해 dict로 반환합니다."""
    lines = text.strip().splitlines()
    labels = [line.split(":", 1)[0].strip() for line in lines if ":" in line]
    return {"두 줄": len(lines) == 2, "항목 순서": labels == REQUIRED_LABELS}


# 형식 사례 두 건(format_rule: exactly_two_lines)을 코드로 판정하고 사람 판정과 비교합니다.
luna_format_rows = []
for row in luna_eval_rows:
    if row["dimension"] == "format" and row.get("format_rule") == "exactly_two_lines":
        checks = check_two_line_format(row["actual_output"])
        code_label = "pass" if all(checks.values()) else "fail"
        luna_format_rows.append({
            "케이스 ID": row["case_id"], "사람 판정": row["human_label"],
            "코드 판정": code_label, "일치": row["human_label"] == code_label, **checks,
        })
display(pd.DataFrame(luna_format_rows))

### 7-3. 질문 충족·근거성 사례를 G-Eval로 확인하기

질문 충족도 3건(L04~L06)과 근거성 3건(L09~L11)에는 4-2에서 만든 `accuracy_ref_metric`(질문 충족과 근거성, 기준 답변 포함)을 적용합니다. 질문 충족도 사례(L04~L06)와 L11의 `expected_output`은 "분위기를 고르거나 구체적인 시작 방법을 안내한다."처럼 기대 행동을 설명한 문장이고, L09·L10은 모범 답 문장입니다. 이번에는 `measure()`를 반복하지 않고 `evaluate()`로 여섯 건을 함께 평가한 뒤, 케이스 이름으로 사람 판정과 연결합니다.

In [ ]:
# 질문 충족·근거성 사례 6건을 LLMTestCase로 만들고 함께 평가합니다.
accuracy_source_by_id = {}
accuracy_eval_cases = []
for row in luna_eval_rows:
    if row["dimension"] in ("question_fulfillment", "grounding"):
        accuracy_source_by_id[row["case_id"]] = row
        accuracy_eval_cases.append(LLMTestCase(
            input=row["input"], actual_output=row["actual_output"],
            expected_output=row["expected_output"], context=row["context"], name=row["case_id"],
        ))

accuracy_eval_result = evaluate(
    test_cases=accuracy_eval_cases,
    metrics=[accuracy_ref_metric],
    async_config=AsyncConfig(max_concurrent=3, throttle_value=0),
    display_config=DisplayConfig(print_results=False, show_indicator=False),
)
luna_accuracy_rows = []
for test_result in accuracy_eval_result.test_results:
    metric_data = test_result.metrics_data[0]
    source_row = accuracy_source_by_id[test_result.name]
    judge_label = "pass" if metric_data.success else "fail"
    luna_accuracy_rows.append({
        "케이스 ID": test_result.name,
        "평가 축": source_row["dimension"],
        "사람 판정": source_row["human_label"],
        "판사 판정": judge_label,
        "일치": source_row["human_label"] == judge_label,
        "점수": round(metric_data.score, 2),
        "판정 이유": metric_data.reason,
    })
display(pd.DataFrame(luna_accuracy_rows).sort_values("케이스 ID"))

**확인할 점:** 사람 판정은 L05(응원만 함)·L10(틀린 시각)·L11(없는 VIP 시각)이 fail, 나머지는 pass입니다. 불일치가 있으면 `판정 이유`가 기준 답변(행동 설명)과 `context`를 어떻게 해석했는지 읽습니다.

### 7-4. 평가 축별 결과를 모아 판사 점검하기

지금까지 루나 15건을 평가 축마다 다른 방법으로 판정했습니다.

| 평가 축 | 사례 | 판정 방법 | 위치 |
|---|---|---|---|
| 존중 표현·편향 | L12~L15 | `ToxicityMetric`·`BiasMetric` | 6절 |
| 페르소나 | L01~L03 | G-Eval(루나 페르소나 일관성) | 7-1 |
| 형식 | L07·L08 | 코드 검사 | 7-2 |
| 질문 충족·근거성 | L04~L06, L09~L11 | G-Eval(질문 충족과 근거성) | 7-3 |

결과를 한 표로 모으고, 교안 01 6-2와 같은 **판사 성적표**(일치율·실패 탐지율·정상 통과율·kappa)를 계산합니다. 코드 검사 결과도 사람 판정과 대조할 자동 판정이므로 `판사 판정` 열에 함께 넣습니다.

In [ ]:
from sklearn.metrics import cohen_kappa_score


# [보조 함수] 판사 성적표: 교안 01 6-2와 같은 네 지표를 계산합니다.
def summarize_judge_results(rows):
    """"사람 판정"·"판사 판정" 키가 있는 행 목록으로 일치율·실패 탐지율·정상 통과율·kappa를 계산합니다."""
    human_labels = [row["사람 판정"] for row in rows]
    judge_labels = [row["판사 판정"] for row in rows]
    judge_on_human_fail = [judge for human, judge in zip(human_labels, judge_labels) if human == "fail"]
    judge_on_human_pass = [judge for human, judge in zip(human_labels, judge_labels) if human == "pass"]
    same_count = sum(human == judge for human, judge in zip(human_labels, judge_labels))
    return {
        "사례 수": len(rows),
        "일치율": round(same_count / len(rows), 2),
        "실패 탐지율": round(judge_on_human_fail.count("fail") / len(judge_on_human_fail), 2),
        "정상 통과율": round(judge_on_human_pass.count("pass") / len(judge_on_human_pass), 2),
        "kappa": round(cohen_kappa_score(human_labels, judge_labels), 2),
    }


# 평가 축별 결과를 같은 열 이름으로 모읍니다. 코드 검사의 "코드 판정"도 자동 판정으로 함께 다룹니다.
luna_check_rows = []
for method, rows, judge_key in [
    ("Toxicity·Bias 지표", social_eval_rows_by_metric, "판사 판정"),
    ("G-Eval 페르소나", luna_persona_rows, "판사 판정"),
    ("코드 형식 검사", luna_format_rows, "코드 판정"),
    ("G-Eval 질문 충족·근거성", luna_accuracy_rows, "판사 판정"),
]:
    for row in rows:
        luna_check_rows.append({
            "케이스 ID": row["케이스 ID"], "판정 방법": method,
            "사람 판정": row["사람 판정"], "판사 판정": row[judge_key],
            "일치": row["사람 판정"] == row[judge_key],
        })

luna_check_table = pd.DataFrame(luna_check_rows).sort_values("케이스 ID").reset_index(drop=True)
display(luna_check_table)
display(pd.DataFrame([summarize_judge_results(luna_check_rows)], index=["루나 15건 자동 평가"]))

**확인할 점:** 이 성적표는 서로 다른 평가 방법의 결과를 모은 것이므로, 수치 하나로 특정 지표의 성능을 말할 수는 없습니다. 축마다 2~3건뿐이라 수치는 참고용이고, `일치`가 False인 줄의 이유를 읽어 기준·사례·평가 방법 중 무엇을 고칠지 정하는 것이 목적입니다. 실무에서는 평가 축별로 사례를 늘려 따로 집계하고, 모델이나 프롬프트를 바꿀 때마다 같은 표를 다시 만듭니다.

### 7-5. 평가 범위표에서 질문을 고르기

**평가 범위표**는 어떤 질문 유형과 실패 상황을 사례에 넣었는지 정리한 표입니다. 실제로 자주 들어오는 질문과 예상되는 오류를 나누어 대표 사례를 고릅니다.

<img src="./images/16_coverage_testcases.png" width="760" alt="루나의 페르소나·질문 충족·형식·근거 상태를 조합해 평가 질문을 설계한다">

루나의 새 질문 6건(`luna_coverage_cases.json`)은 평가 축과 **근거 상태**를 조합해 골랐습니다.

| 근거 상태 | 뜻 | 사례 |
|---|---|---|
| 준비된 기준 | 정답이 하나가 아니어서 기준 답변으로 판단 | C1 페르소나, C2 질문 충족 |
| 정확한 형식 | 지정한 형식을 지켜야 함 | C3 |
| 공식 정보 있음 | 공지에서 답을 찾을 수 있음 | C4 |
| 정보 없음 | 공지에 답이 없어 확인할 수 없다고 해야 함 | C5 |
| 틀린 전제 | 질문 속 정보가 공지와 달라 바로잡아야 함 | C6 |

기준 답변과 근거가 각 상황을 제대로 반영하는지 읽습니다.

In [ ]:
# 평가 질문과 해당 근거 ID를 커버리지 표로 확인합니다.
coverage_rows = read_json("luna_coverage_cases.json")
coverage_table = []
for row in coverage_rows:
    coverage_table.append({
        "케이스 ID": row["case_id"], "평가 축": row["dimension"],
        "근거 상태": row["evidence_state"], "질문": row["question"],
        "기준 답변": row["expected_output"],
        "근거 ID": ", ".join(row["context_ids"]),
    })
display(pd.DataFrame(coverage_table))

### 7-6. LangChain `batch()`로 여러 답변 만들기

LangChain의 `batch()`는 서로 독립된 질문을 한꺼번에 보내 답변을 만듭니다. 여기서는 루나에게 평가 범위표의 질문 여섯 개를 보냅니다. 검색기는 실행하지 않고 미리 준비한 루나 설정과 공지를 프롬프트에 넣습니다. 평가 사례에는 질문에 맞는 근거만 연결합니다.

`max_concurrency`는 한 번에 처리할 요청 수를 제한합니다. 병렬 처리는 기다리는 시간을 줄일 수 있지만 API 요청 수와 비용을 줄이지는 않습니다. 이는 공급자가 제공하는 별도의 대량 요청 API와도 다릅니다. [LangChain `batch()` 공식 문서](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable/batch)

In [ ]:
# 루나에게 전달할 전체 설정 문장을 ID와 함께 구성합니다.
guide_lines = [guide["notice"], guide["evidence_scope"]]
for sentence in guide["sentences"]:
    guide_lines.append(sentence["id"] + " " + sentence["text"])
guide_text = "\n".join(guide_lines)

# 루나는 설정과 공지에 없는 사실을 지어내지 않도록 안내받습니다.
luna_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 팬을 존중하는 따뜻하고 경쾌한 버추얼 아이돌 루나입니다. "
               "아래 수업용 설정과 공지에 적힌 내용만 사실로 안내하세요. "
               "없는 정보는 추측하지 말고 확인할 수 없다고 말하세요. 두세 문장으로 답하세요."),
    ("human", "루나 설정과 공지:\n{guide}\n\n질문: {question}"),
])
luna_chain = luna_prompt | answer_llm

In [ ]:
# ✏️ 부분 작성: `...` 자리를 채운 뒤 실행합니다.
# 1. 질문과 설정을 답변 입력으로 묶습니다.
luna_inputs = []
for row in coverage_rows:
    luna_inputs.append(...)  # luna_prompt의 두 변수(guide, question)를 담은 dict
# 2. LangChain batch로 여섯 질문을 함께 보냅니다(max_concurrency=4).
luna_generated_responses = ...

# 3. 결과는 입력 순서와 같으므로 원래 케이스 ID와 근거를 붙입니다.
coverage_test_cases = []
coverage_answer_rows = []
for row, answer_message in zip(coverage_rows, luna_generated_responses):
    answer = answer_message.text
    coverage_test_cases.append(...)  # LLMTestCase: context에는 row["context_ids"]의 문장 목록을 넣습니다.
    coverage_answer_rows.append({
        "케이스 ID": row["case_id"], "평가 축": row["dimension"],
        "근거 상태": row["evidence_state"], "질문": row["question"],
        "기준 답변": row["expected_output"], "루나의 답": answer,
    })
display(pd.DataFrame(coverage_answer_rows))

**확인할 점:** `정보 없음`과 `틀린 전제` 사례에서 루나가 없는 정보를 만들어 내지 않는지, 질문의 전제를 바로잡는지 봅니다. 생성 답변은 실행마다 달라질 수 있습니다.

## 8. `evaluate()`로 여러 사례를 함께 평가하기

DeepEval의 `evaluate()`는 여러 평가 사례에 지정한 지표를 적용하고 결과를 모읍니다. 7-6에서 만든 여섯 답변에는 **질문 충족·근거성 G-Eval**(기준 답변 포함)과 **근거 모순(Hallucination) 지표**를 적용합니다. 형식 사례 C3은 7-2의 형식 검사 함수로도 확인합니다. 서로 다른 지표 점수는 더해 하나의 품질 점수로 만들지 않습니다.

존중 표현·편향 지표는 6절에서 그 기준에 맞는 사례에 적용했습니다. 실무에서는 `ToxicityMetric`·`BiasMetric` 같은 안전 지표를 모든 응답에 안전장치로 함께 돌리기도 합니다. 이 실습에서는 지표의 목적을 분명히 하고 요청 수를 줄이기 위해 나눠 적용합니다.

`AsyncConfig(max_concurrent=4)`는 동시에 처리할 평가 사례 수를 제한합니다. 한 사례에 지표를 여러 개 적용하면 지표들이 함께 실행되므로 모델 요청 수는 사례 수보다 많을 수 있습니다.

In [ ]:
# 질문 충족·근거성 G-Eval과 근거 모순 지표를 여섯 답변에 함께 실행합니다.
metrics = [accuracy_ref_metric, hallucination_metric]
coverage_result = evaluate(
    test_cases=coverage_test_cases,
    metrics=metrics,
    async_config=AsyncConfig(max_concurrent=4, throttle_value=0),
    display_config=DisplayConfig(print_results=False, show_indicator=False),
)

In [ ]:
# 케이스·지표 조합마다 점수·통과 여부·이유를 한 행으로 만듭니다.
coverage_eval_rows = []
for test_result in coverage_result.test_results:
    for metric_data in test_result.metrics_data:
        coverage_eval_rows.append({
            "케이스 ID": test_result.name, "지표": metric_data.name,
            "점수": round(metric_data.score, 2), "통과": metric_data.success,
            "이유": metric_data.reason,
        })
coverage_eval_table = pd.DataFrame(coverage_eval_rows)
display(coverage_eval_table.pivot(index="케이스 ID", columns="지표", values="통과"))


# 형식 사례 C3은 7-2의 코드 검사로도 확인합니다.
for row, answer_row in zip(coverage_rows, coverage_answer_rows):
    if row["dimension"] == "format":
        print(row["case_id"], "형식 검사:", check_two_line_format(answer_row["루나의 답"]))

In [ ]:
# 통과하지 못한 지표 결과를 모아 판정 이유를 읽습니다.
coverage_failed_rows = []
for row in coverage_eval_rows:
    if not row["통과"]:
        coverage_failed_rows.append(row)
print("실패한 케이스·지표 수:", len(coverage_failed_rows))
display(pd.DataFrame(coverage_failed_rows))

**확인할 점:** 실패한 케이스를 평가 범위표와 대조해 어떤 근거 상태(정보 없음, 틀린 전제 등)에서 문제가 생겼는지 찾습니다. 그다음 `이유`와 답변 원문을 함께 읽고, 실제 답변의 오류인지, 평가 기준·기준 답변·`context` 구성이 부적절한 탓인지 구분합니다. Hallucination 점수는 낮을수록 좋다는 점도 다시 확인합니다. 여섯 사례에서 실패가 없어도 다른 입력까지 통과한다는 뜻은 아닙니다.

## 9. 함께 따라하기: 루나 굿즈 스토어 상담 답변 평가하기

루나 굿즈 스토어 상담 챗봇이 응원봉·랜덤 포토카드·주문 제작 굿즈의 교환·반품 질문에 답한다고 가정합니다. 이 실습은 페르소나 말투가 아니라 **정책 조건을 빠짐없이 안내하는지, 확인되지 않은 환불·발송을 약속하지 않는지, 필요한 다음 절차를 알려 주는지** 평가합니다.

사람이 미리 확인한 상담 기록 여덟 건의 v1·v2 답변 16개로 평가 기준을 살핍니다. 그다음 다른 질문 여섯 건에 답을 새로 만들고 DeepEval로 평가합니다. 모든 정책과 상담 기록은 수업용 가상 자료이며 실제 주문을 조회하거나 처리하지 않습니다.

**실습 순서**

1. 가상 정책과 상담 기록을 읽습니다.
2. 정책 기준을 G-Eval로 정해 기록된 답변과 사람 판정을 비교합니다(교안 01 7-4와 같은 자료).
3. 새 질문 여섯 건에 답을 생성하고 정책 근거를 연결합니다.
4. 실패 이유를 읽고 답변 문제인지 평가 기준 문제인지 확인합니다.

> `✏️ 직접 설계`·`✏️ 부분 작성` 표시가 있는 셀은 학생본에서 `...` 자리를 채웁니다. 완성 코드는 `정답/교안_02_정답.ipynb`에 있습니다.

### 9-1. 굿즈 정책과 상담 기록 읽기

정책에는 단순 변심의 기한과 상품 상태, 파손·오배송 접수 기한과 사진, 랜덤 굿즈와 주문 제작 상품의 예외, 주문 조회 한계와 환불 기간을 확정할 수 없다는 원칙이 있습니다. `policy_context()`는 선택한 정책 조항을 근거 문장 목록으로 바꿉니다.

In [ ]:
# 가상 정책·사람 판정이 있는 상담 로그·새 평가 질문을 읽습니다.
store_policies = read_json("store_policies.json")
store_policy_by_id = {policy["policy_id"]: policy for policy in store_policies}
store_log = read_json("store_chatbot_log.json")
store_coverage = read_json("store_coverage_cases.json")

# 필요한 정책의 조항을 나눠 근거 목록으로 돌려줍니다.
def policy_context(policy_ids):
    """정책 ID에 해당하는 제목과 조항을 근거 문장 목록으로 반환합니다."""
    context = []
    for policy_id in policy_ids:
        policy = store_policy_by_id[policy_id]
        context.append(f"{policy_id} {policy['title']}")
        for section, text in policy["sections"].items():
            context.append(f"{policy_id} {section}: {text}")
    return context

print("정책 수:", len(store_policies))
print("사람 판정 상담 수:", len(store_log))
print("새 평가 질문 수:", len(store_coverage))
print("P2 파손·오배송 정책 예시:", *policy_context(["P2"]), sep="\n")

### 9-2. 정책 준수 기준을 G-Eval로 적기

정책 준수 지표는 **질문에 맞는 정책 조건과 다음 절차를 안내했는지** 평가합니다. 정책 근거와 사람이 확인한 기준 답변을 함께 제공하고, 필수 조건 누락·정책과 반대되는 안내·확인되지 않은 약속·다음 문의 방법을 살핍니다.

| 설정 | 작성할 내용 |
|---|---|
| `name` | `루나 스토어 정책 준수` |
| `evaluation_steps` | 질문 상황, 정책의 조건·예외·기한·비용·다음 절차를 차례로 확인 |
| `evaluation_params` | 질문·평가할 답·기준 답변·근거 문장 |
| 말투·길이 | 정책 준수 점수에 포함하지 않음 |
| `threshold` | 통과 표시를 정하는 값. 사람 판정과 비교해 조정 |

아래 코드 빈칸에 위 기준을 단계로 적습니다. `threshold`는 통과 표시의 경계이지, 서비스 출시를 보증하는 값은 아닙니다.

In [ ]:
# ✏️ 직접 설계: 위 표를 보고 정책 준수 기준을 G-Eval 판단 단계로 적습니다.
store_policy_metric = GEval(
    name="루나 스토어 정책 준수",
    evaluation_steps=[
        "input에서 고객 상황과 요청을 확인한다.",
        ...,  # 교환·반품 가능 여부와 조건을 context 및 expected_output과 대조하는 단계
        ...,  # 기한·상품 상태·사진·문의 절차·비용 분담 같은 필수 내용 누락을 감점하는 단계
        ...,  # 정책에 없는 환불 기간·즉시 발송·환불 확정·주문 조회 결과 약속을 크게 감점하는 단계
        ...,  # 고객이 이어서 할 접수 방법을 안내했는지 확인하는 단계
        ...,  # 말투·길이는 반영하지 않고, 표현이 달라도 정책상 같은 의미면 인정하는 단계
    ],
    evaluation_params=...,  # 질문·평가할 답·기준 답변·근거 문장(SingleTurnParams 네 개)
    model=judge_llm,
    threshold=0.8,  # 수업용 시작값이며 사람 판정 사례를 보고 조정합니다.
)

### 9-3. 기록된 16개 답변을 사람 판정과 비교하기

상담 질문 여덟 건의 v1·v2 답을 각각 `LLMTestCase`로 만듭니다. 질문 ID와 버전을 이름으로 붙여 판사 결과와 사람 판정을 연결합니다. 두 답 모두 실패인 사례와 둘 다 통과해 사람도 우열을 가리지 않은 사례가 있습니다. 상대 비교 결과와 각 답의 통과 여부가 다른 판단임을 확인합니다.

In [ ]:
# 각 정책 로그에서 v1·v2를 각각 테스트 케이스로 만듭니다.
store_log_cases = []
store_label_by_name = {}
for row in store_log:
    context = policy_context(row["policy_ids"])
    for version in ["v1", "v2"]:
        answer = row[f"answer_{version}"]
        case_name = f"{row['question_id']}-{version}"
        store_log_cases.append(LLMTestCase(
            input=row["question"], actual_output=answer,
            expected_output=row["reference"], context=context, name=case_name,
        ))
        store_label_by_name[case_name] = row[f"label_{version}"]

print("로그 테스트 케이스:", len(store_log_cases))

In [ ]:
# 기록된 16개 응답을 비동기로 판정하고 사람 판정과 나란히 둡니다.
store_log_result = evaluate(
    test_cases=store_log_cases,
    metrics=[store_policy_metric],
    async_config=AsyncConfig(max_concurrent=4, throttle_value=0),
    display_config=DisplayConfig(print_results=False, show_indicator=False),
)
store_log_rows = []
for test_result in store_log_result.test_results:
    metric = test_result.metrics_data[0]
    judge_label = "pass" if metric.success else "fail"
    store_log_rows.append({
        "케이스 ID": test_result.name,
        "사람 판정": store_label_by_name[test_result.name],
        "판사 판정": judge_label,
        "일치": store_label_by_name[test_result.name] == judge_label,
        "점수": round(metric.score, 2),
        "이유": metric.reason,
    })
display(pd.DataFrame(store_log_rows))


# 교안 01 6-2와 같은 판사 성적표로 요약합니다.
display(pd.DataFrame([summarize_judge_results(store_log_rows)], index=["정책 준수 G-Eval"]))

**확인할 점:** 사람 판정과 다른 줄을 찾아 정책 문장·기준 답변·G-Eval 이유를 함께 읽습니다. `사람 fail / 판사 pass`는 놓친 실패이고, `사람 pass / 판사 fail`은 헛경보입니다. 점수가 맞아도 이유가 정책 조건을 잘못 읽었다면 기준을 수정해야 합니다.


교안 01 7-4의 LangChain 판사 성적표와 비교해, 같은 자료에서 두 방식 모두 어려워한 사례가 있는지도 살펴봅니다.

### 9-4. 새 질문의 평가 범위 확인하기

새 질문 여섯 건은 단순 변심, 랜덤 굿즈, 파손 접수 기한 초과, 주문 제작 예외, 환불 기간 미정, 주문 조회 한계를 다룹니다. 질문별로 어떤 정책 조건과 예외를 확인할지 살펴봅니다.

In [ ]:
# 새 질문을 정책 ID·필수 안내(required_points)와 함께 확인합니다.
store_coverage_table = []
for row in store_coverage:
    store_coverage_table.append({
        "케이스 ID": row["case_id"],
        "상황": row["category"],
        "질문": row["question"],
        "확인할 점": " / ".join(row["required_points"]),
        "정책": ", ".join(row["policy_ids"]),
    })
display(pd.DataFrame(store_coverage_table))

### 9-5. LangChain `batch()`로 상담 답변 만들기

상담 챗봇은 주문 시스템에 연결되어 있지 않습니다. 따라서 주문 조회나 환불 처리를 했다고 말하지 않고, 정책에 적힌 내용과 필요한 다음 문의 절차만 안내하도록 설정합니다. 여섯 질문을 `batch()`로 묶어 답을 만듭니다.

In [ ]:
# 챗봇에 전달할 가상 스토어 정책을 읽기 쉬운 문장으로 만듭니다.
store_policy_lines = []
for policy in store_policies:
    store_policy_lines.append(f"[{policy['policy_id']}] {policy['title']}")
    for section, text in policy["sections"].items():
        store_policy_lines.append(f"{section}: {text}")
store_policy_text = "\n".join(store_policy_lines)

# 주문 조회·환불 실행을 할 수 없다고 명시한 상담 챗봇 프롬프트입니다.
store_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 루나 굿즈 스토어 상담 챗봇입니다. 제공한 수업용 정책만 근거로 답하세요. "
               "정책에 없는 환불 기간, 교환 확정, 즉시 발송을 약속하지 마세요. "
               "주문 조회나 환불 실행을 했다고 말하지 말고 필요한 다음 절차를 안내하세요. 세 문장 이내로 답하세요."),
    ("human", "정책:\n{policies}\n\n고객 질문: {question}"),
])
store_chain = store_prompt | answer_llm

In [ ]:
# ✏️ 부분 작성: `...` 자리를 채운 뒤 실행합니다.
# 1. 정책과 고객 질문 6건을 답변 입력으로 묶습니다.
store_inputs = []
for row in store_coverage:
    store_inputs.append(...)  # store_prompt의 두 변수(policies, question)를 담은 dict

# 2. LangChain batch로 여섯 질문을 함께 보냅니다(max_concurrency=4).
store_generated_responses = ...

# 3. 모델 응답에 해당 질문의 정책 근거만 붙여 테스트 케이스를 만듭니다.
store_coverage_cases = []
store_answer_rows = []
for row, answer_message in zip(store_coverage, store_generated_responses):
    answer = answer_message.text
    store_coverage_cases.append(...)  # LLMTestCase: context에는 policy_context(row["policy_ids"])를 넣습니다.
    store_answer_rows.append({"케이스 ID": row["case_id"], "상황": row["category"],
                             "질문": row["question"], "기준 답변": row["reference"],
                             "상담 답변": answer})

display(pd.DataFrame(store_answer_rows))

**확인할 점:** 파손 접수 기한이 지난 사례, 환불 기간을 묻는 사례, 주문 조회를 요청하는 사례에서 상담 답이 할 수 있는 일과 없는 일을 구분하는지 확인합니다. 모델이 새로 만든 응답은 실행할 때마다 달라질 수 있습니다.


T3은 접수 기한(14일)이 지난 사례라 P2만으로 처리 결과를 확정할 수 없습니다. 기준 답변의 '1:1 문의로 개별 확인 요청'은 P2의 접수 경로를 근거로 하므로, 답변이 기한을 알리면서 환불을 확정하지 않는지 봅니다.

### 9-6. 두 지표로 상담 답변 평가하기

같은 답변에 정책 준수 G-Eval과 준비된 정책 근거와의 모순 지표를 적용합니다. G-Eval은 조건 누락과 다음 절차 안내도 확인합니다. 근거 모순 지표는 답변과 제공한 정책 문장이 서로 모순되는지를 봅니다. 정책에 없는 약속은 근거와 모순되지 않을 수 있으므로 G-Eval의 판정 이유도 읽습니다.

In [ ]:
# ✏️ 부분 작성: `...` 자리를 채운 뒤 실행합니다.
# 정책 준수와 모순 여부를 두 지표로 평가합니다.
store_result = evaluate(
    test_cases=...,  # 9-5에서 만든 store_coverage_cases
    metrics=...,  # 정책 준수 G-Eval(store_policy_metric)과 hallucination_metric
    async_config=AsyncConfig(max_concurrent=4, throttle_value=0),
    display_config=DisplayConfig(print_results=False, show_indicator=False),
)

# 케이스·지표마다 점수와 통과 여부, 이유를 한 행에 기록합니다.
store_eval_rows = []
for test_result in store_result.test_results:
    for metric_data in test_result.metrics_data:
        store_eval_rows.append({
            "케이스 ID": test_result.name, "지표": metric_data.name,
            "점수": round(metric_data.score, 2), "통과": metric_data.success,
            "이유": metric_data.reason,
        })
store_eval_table = pd.DataFrame(store_eval_rows)
display(store_eval_table.pivot(index="케이스 ID", columns="지표", values="통과"))
# 실패 행을 상담 답변과 정책 조항에 대조합니다.
display(store_eval_table[store_eval_table["통과"] == False])

**확인할 점:** 평가 실패가 실제 상담 답변의 문제인지, 기준 답변·필수 조건·정책 근거가 잘못 구성된 탓인지 구분합니다. 판정 이유와 해당 상담 답을 함께 읽습니다. 일부 케이스에서 모두 통과해도 다른 질문까지 맞는다는 보증은 아닙니다.

## 10. 핵심 정리

- **평가 사례(`LLMTestCase`):** 질문(`input`)·평가할 답(`actual_output`)·기준 답변(`expected_output`)·근거 문장(`context`)을 한 건으로 묶습니다. 기준 답변에는 모범 답 문장이나 기대 행동 설명을 넣을 수 있습니다.
- **G-Eval:** 우리가 정한 기준을 판단 단계(`evaluation_steps`)로 적고, 점수와 판정 이유를 확인합니다. rubric으로 점수 구간의 뜻을 정할 수 있으며, `threshold`는 통과 표시의 경계값일 뿐입니다.
- **지표는 목적별로:** 근거 모순(Hallucination)은 답과 근거 사이의 모순, Toxicity는 모욕·위협, Bias는 집단 고정관념을 봅니다. DeepEval 4.1.1에서 세 지표는 점수가 낮을수록 좋습니다. 지표가 겨냥하지 않은 결함은 놓칠 수 있으며, 어느 하나도 답변 품질 전체를 보증하지 않습니다.
- **규칙은 코드로, 맥락은 판사로:** 줄 수·항목 순서처럼 분명한 규칙은 코드로 검사하고, 말투·정책 적용처럼 맥락이 필요한 기준은 판사 모델로 평가합니다.
- **사람 판정과 대조:** 평가 축마다 알맞은 방법으로 판정한 뒤, 교안 01과 같은 성적표(일치율·실패 탐지율·정상 통과율·kappa)와 불일치 사례의 이유로 판사를 점검합니다.
- **평가 범위표:** 질문 유형과 근거 상태를 조합해 대표 사례를 고릅니다. 적은 사례만으로 모든 질문의 품질을 알 수는 없습니다.
- **`evaluate()`와 `batch()`:** `evaluate()`는 평가 사례를 여러 지표로 채점하고, LangChain `batch()`는 여러 답변을 함께 만듭니다. 병렬 처리는 대기 시간을 줄일 수 있지만 요청 수와 비용을 줄이지 않습니다.
- **RAG:** 같은 평가 방법을 RAG 답변에도 쓸 수 있습니다. 이 실습은 검색을 실행하지 않고 미리 준비한 근거로 답변을 평가했습니다.